# 04 — Concept Intervention  ·  Build step 04 (pre-G3) · Gate-G3 input · Novelty I8 (clinician demo)

Trains the `independent` concept bottleneck and measures how the diagnosis responds when each concept is set from a
low to a high counterfactual value (does the diagnosis actually depend on the clinical concepts?), plus a directed
"clinician override" (e.g. assert a strong crackle → does COPD probability move as expected?).

Needs `concepts_all.npz` (nb01), `M2_features.npy` (step 02). Output: `intervention_report.json`.

In [1]:
import sys, os
OWMTL_PKG = "/kaggle/input/datasets/barshonbasak/owmtl-package"
sys.path.insert(0, OWMTL_PKG)
import numpy as np, json
from collections import defaultdict
from owmtl.icbhi_data import KNOWN_DISEASES
OUT_DIR = "/kaggle/working"
Z = np.load("/kaggle/input/datasets/barshonbasak/concepts-all/concepts_all.npz", allow_pickle=True)
X = Z["X"].astype("float32"); patient = Z["patient"]; split = Z["split"]
diagnosis = Z["diagnosis"]; concept_names = list(Z["concept_names"])

# The `independent` bottleneck consumes concepts only, so M2 features are optional here --
# but say so out loud. This used to be a bare try/except that silently substituted zeros.
M2_PATH = "/kaggle/input/datasets/barshonbasak/m2-features/M2_features.npy"
if os.path.exists(M2_PATH):
    F = np.load(M2_PATH).astype("float32")
    assert len(F) == len(X), f"M2 features ({len(F)}) not aligned to cycles ({len(X)})"
    HAVE_FEATS = True
else:
    print(f"NOTE: {M2_PATH} not found -> features stubbed with zeros. Fine for mode="
          "'independent' (concepts only); any other mode would be meaningless.")
    F = np.zeros((len(X), 1), "float32"); HAVE_FEATS = False

# patient-level aggregation (known-class disease task)
lab_map = {d: i for i, d in enumerate(KNOWN_DISEASES)}
by = defaultdict(list)
for i in range(len(X)):
    if diagnosis[i] in lab_map: by[patient[i]].append(i)
pids, Xp, Fp, yp, spp = [], [], [], [], []
for pid, idxs in by.items():
    idxs = np.array(idxs); pids.append(pid)
    Xp.append(X[idxs].mean(0)); Fp.append(F[idxs].mean(0))
    yp.append(lab_map[diagnosis[idxs[0]]]); spp.append(split[idxs[0]])
pids, Xp, Fp, yp, spp = map(np.array, (pids, np.stack(Xp), np.stack(Fp), yp, spp))

# STANDARDISE on train only -- intervention sensitivity is compared ACROSS concepts, so
# it is meaningless in raw units (low_high_freq_ratio spans 0.018-18910, crackle_presence
# spans [0,1]). The 2026-08-26 ranking was a units artifact. Percentile counterfactuals
# below are now in z-units, i.e. the same perturbation size for every concept.
from sklearn.preprocessing import StandardScaler
_tr = spp == "train"
Xp = StandardScaler().fit(Xp[_tr]).transform(Xp).astype("float32")
Fp = StandardScaler().fit(Fp[_tr]).transform(Fp).astype("float32")
print("patients", len(pids), "| classes", KNOWN_DISEASES, "| M2 features:", HAVE_FEATS)

patients 104 | classes ['COPD', 'Healthy', 'URTI'] | M2 features: True


In [2]:
import torch
from owmtl.bottleneck import ConceptBottleneck
from owmtl.intervention import intervention_sensitivity, directed_intervention, summarize
tr = spp == "train"; K = len(KNOWN_DISEASES)
m = ConceptBottleneck(Xp.shape[1], Fp.shape[1], K, mode="independent", hidden=64)
xc, xf, ty = torch.tensor(Xp), torch.tensor(Fp), torch.tensor(yp).long()
cw = torch.tensor([(yp[tr]==k).sum() for k in range(K)], dtype=torch.float); cw = (cw.sum()/(cw+1e-6)); cw=cw/cw.sum()*K
opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=1e-4)
trt = torch.tensor(tr)
for _ in range(200):
    opt.zero_grad(); lg,ch = m(xf[trt], xc[trt]); l = m.loss(lg, ty[trt], ch, xc[trt], class_weight=cw); l.backward(); opt.step()
rows = intervention_sensitivity(m, Fp, Xp, concept_names)
print(summarize(rows, top_k=6))

Concept intervention sensitivity (diagnosis responds to which concepts):
  inspiratory_energy_fraction sensitivity=0.438
  rhonchi_presence         sensitivity=0.416
  papr_db                  sensitivity=0.394
  crackle_presence         sensitivity=0.263
  transient_timing_centroid sensitivity=0.165
  wheeze_duration_ratio    sensitivity=0.125
  (total responsiveness across concepts = 2.352; ~0 means the model ignores the concept layer)


In [3]:
# directed clinician-style interventions: assert a strong crackle / strong wheeze
name_idx = {n:i for i,n in enumerate(concept_names)}
hi = np.percentile(Xp, 90, axis=0)
report = {"gate": "G3_input", "sensitivity": rows, "directed": {}}
for concept, target in [("crackle_presence", "COPD"), ("wheeze_presence", "COPD")]:
    if target in KNOWN_DISEASES:
        di = directed_intervention(m, Fp, Xp, name_idx[concept], hi[name_idx[concept]],
                                   target_class=KNOWN_DISEASES.index(target))
        report["directed"][f"{concept}->{target}"] = di
        print(concept, "->", target, di)
with open(os.path.join(OUT_DIR, "intervention_report.json"), "w") as fh: json.dump(report, fh, indent=2)

crackle_presence -> COPD {'concept_idx': 0, 'new_value': 1.54814612865448, 'target_class': 0, 'mean_delta_p': 0.1398, 'frac_increased': 0.894}
wheeze_presence -> COPD {'concept_idx': 4, 'new_value': 1.381134271621704, 'target_class': 0, 'mean_delta_p': -0.0008, 'frac_increased': 0.433}


**Interpretation.** High total sensitivity + the *clinically-expected* concepts ranking top ⇒ the diagnosis is
genuinely driven by the clinical concepts (a Path-A positive). Near-zero sensitivity ⇒ the model ignores the concept
layer. This is the third Gate-G3 input; the A/B decision is made at G3, not here.